In [ ]:

import pandas as pd
import faiss
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

# Model names
SAPBERT_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
BILORD_MODEL = "FremyCompany/BioLORD-2023"  # Example Bilord model

# Paths

SAPBERT_INDEX = r"C:\Users\UNegi\Downloads\icd10_faiss_sapbert_from_pubmedbert.index"
SAPBERT_META = r"C:\Users\UNegi\Downloads\icd10_metadata_sapbert_from_pubmedbert.xlsx"
BILORD_INDEX = r"C:\Users\UNegi\Documents\Project\makethon\icd10_faiss_biolord.index"
BILORD_META = r"C:\Users\UNegi\Documents\Project\makethon\icd10_metadata_biolord.xlsx"

# Load models
sap_tokenizer = AutoTokenizer.from_pretrained(SAPBERT_MODEL)
sap_model = AutoModel.from_pretrained(SAPBERT_MODEL).to(DEVICE).eval()

bilord_tokenizer = AutoTokenizer.from_pretrained(BILORD_MODEL)
bilord_model = AutoModel.from_pretrained(BILORD_MODEL).to(DEVICE).eval()

# Load FAISS indexes
sap_index = faiss.read_index(SAPBERT_INDEX)
bilord_index = faiss.read_index(BILORD_INDEX)

# Load metadata
sap_meta_df = pd.read_excel(SAPBERT_META)
bilord_meta_df = pd.read_excel(BILORD_META)

sap_metadata = {i: {"code": row["Code"], "description": row["Description"]} for i, row in sap_meta_df.iterrows()}
bilord_metadata = {i: {"code": row["Code"], "description": row["Description"]} for i, row in bilord_meta_df.iterrows()}

# Utility functions
def mean_pooling(last_hidden_state, attention_mask):
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)

def embed_texts(texts, tokenizer, model):
    all_embs = []
    with torch.no_grad():
        for i in range(0, len(texts), BATCH_SIZE):
            batch = texts[i:i+BATCH_SIZE]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
            out = model(**enc)
            emb = mean_pooling(out.last_hidden_state, enc["attention_mask"])
            emb = emb.cpu().numpy()
            all_embs.append(emb)
    embs = np.vstack(all_embs).astype("float32")
    embs /= np.linalg.norm(embs, axis=1, keepdims=True)  # L2 normalization
    return embs

def normalize_scores(scores):
    arr = np.array(scores)
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

def search_model(query, index, tokenizer, model, metadata, top_k=100):
    q_emb = embed_texts([query], tokenizer, model)
    D, I = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        meta = metadata[idx]
        results.append({"code": meta["code"], "description": meta["description"], "score": float(score), "id": idx})
    return results

def late_fusion(query, alpha=0.5, beta=0.5, top_k=10):
    sap_results = search_model(query, sap_index, sap_tokenizer, sap_model, sap_metadata, top_k)
    bilord_results = search_model(query, bilord_index, bilord_tokenizer, bilord_model, bilord_metadata, top_k)

    # Normalize scores
    sap_norm = normalize_scores([r["score"] for r in sap_results])
    bilord_norm = normalize_scores([r["score"] for r in bilord_results])

    # Map normalized scores back
    for i, r in enumerate(sap_results): r["norm_score"] = sap_norm[i]
    for i, r in enumerate(bilord_results): r["norm_score"] = bilord_norm[i]

    # Merge and fuse
    combined = {}
    for r in sap_results:
        combined[r["code"]] = alpha * r["norm_score"]
    for r in bilord_results:
        combined[r["code"]] = combined.get(r["code"], 0) + beta * r["norm_score"]

    # Sort by combined score
    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    return ranked

# Example usage
query = "Type 2 diabetes mellitus"
final_results = late_fusion(query)
print(final_results)


c:\Users\UNegi\Documents\Project\makethon\.health\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\UNegi\Documents\Project\makethon\.health\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\UNegi\.cache\huggingface\hub\models--FremyCompany--BioLORD-2023. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administ

[('E1165   ', np.float64(0.8274627073660845)), ('E118    ', np.float64(0.5539101259006698)), ('E1169   ', np.float64(0.24957158713971214)), ('E138    ', np.float64(0.16478261632257504)), ('E108    ', np.float64(0.14551495537892128)), ('E10A2   ', np.float64(0.14175726922573964)), ('Z833    ', np.float64(0.11383999330366278)), ('E119    ', np.float64(0.07964585819450393)), ('O2412   ', np.float64(0.06995515917332955)), ('E1369   ', np.float64(0.044176858845672365)), ('O24112  ', np.float64(0.04142952458569181)), ('R7309   ', np.float64(0.034364734338868025)), ('O24119  ', np.float64(0.023197625447755322)), ('E1144   ', np.float64(0.02203953568386787)), ('O24912  ', np.float64(0.01418560432512111)), ('R7301   ', np.float64(0.012938269944676313)), ('E1141   ', np.float64(0.0)), ('E1365   ', np.float64(0.0))]


In [ ]:
import networkx as nx
import pickle

import pandas as pd
import faiss
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

# Model names
SAPBERT_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
BILORD_MODEL = "FremyCompany/BioLORD-2023"  # Example Bilord model

# Paths

SAPBERT_INDEX = r"C:\Users\UNegi\Downloads\icd10_faiss_sapbert_from_pubmedbert.index"
SAPBERT_META = r"C:\Users\UNegi\Downloads\icd10_metadata_sapbert_from_pubmedbert.xlsx"
BILORD_INDEX = r"C:\Users\UNegi\Documents\Project\makethon\icd10_faiss_biolord.index"
BILORD_META = r"C:\Users\UNegi\Documents\Project\makethon\icd10_metadata_biolord.xlsx"

# Load models
sap_tokenizer = AutoTokenizer.from_pretrained(SAPBERT_MODEL)
sap_model = AutoModel.from_pretrained(SAPBERT_MODEL).to(DEVICE).eval()

bilord_tokenizer = AutoTokenizer.from_pretrained(BILORD_MODEL)
bilord_model = AutoModel.from_pretrained(BILORD_MODEL).to(DEVICE).eval()

# Load FAISS indexes
sap_index = faiss.read_index(SAPBERT_INDEX)
bilord_index = faiss.read_index(BILORD_INDEX)

# Load metadata
sap_meta_df = pd.read_excel(SAPBERT_META)
bilord_meta_df = pd.read_excel(BILORD_META)

sap_metadata = {i: {"code": row["Code"], "description": row["Description"]} for i, row in sap_meta_df.iterrows()}
bilord_metadata = {i: {"code": row["Code"], "description": row["Description"]} for i, row in bilord_meta_df.iterrows()}

# Utility functions
def mean_pooling(last_hidden_state, attention_mask):
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)

def embed_texts(texts, tokenizer, model):
    all_embs = []
    with torch.no_grad():
        for i in range(0, len(texts), BATCH_SIZE):
            batch = texts[i:i+BATCH_SIZE]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
            out = model(**enc)
            emb = mean_pooling(out.last_hidden_state, enc["attention_mask"])
            emb = emb.cpu().numpy()
            all_embs.append(emb)
    embs = np.vstack(all_embs).astype("float32")
    embs /= np.linalg.norm(embs, axis=1, keepdims=True)  # L2 normalization
    return embs

def normalize_scores(scores):
    arr = np.array(scores)
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

def search_model(query, index, tokenizer, model, metadata, top_k=100):
    q_emb = embed_texts([query], tokenizer, model)
    D, I = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        meta = metadata[idx]
        results.append({"code": meta["code"], "description": meta["description"], "score": float(score), "id": idx})
    return results
# Load ICD hierarchy graph

with open("icd_code_hierarchy.pkl", "rb") as f:
    node_dict = pickle.load(f)

root_node = node_dict['root']

def build_networkx_graph(root_node):
    graph = nx.Graph()
    def add_nodes_edges(node):
        graph.add_node(node.icd_code, description=node.description)
        for child in node.get_children():
            graph.add_edge(node.icd_code, child.icd_code)
            add_nodes_edges(child)
    add_nodes_edges(root_node)
    return graph

def compute_graph_cohesion(graph, candidate_codes):
    # Compute pairwise proximity among candidates
    proximity_scores = {code: 0 for code in candidate_codes}
    for i, code1 in enumerate(candidate_codes):
        for j, code2 in enumerate(candidate_codes):
            if i != j and code1 in graph and code2 in graph:
                try:
                    dist = nx.shortest_path_length(graph, source=code1, target=code2)
                    proximity_scores[code1] += 1 / (dist + 1)
                except nx.NetworkXNoPath:
                    continue
    # Normalize by number of neighbors
    for code in proximity_scores:
        proximity_scores[code] /= (len(candidate_codes) - 1)
    return proximity_scores

def late_fusion_with_graph_cohesion(query, alpha=0.5, beta=0.5, gamma=0.3, top_k=60):
    # Late fusion
    sap_results = search_model(query, sap_index, sap_tokenizer, sap_model, sap_metadata, top_k)
    bilord_results = search_model(query, bilord_index, bilord_tokenizer, bilord_model, bilord_metadata, top_k)

    sap_norm = normalize_scores([r["score"] for r in sap_results])
    bilord_norm = normalize_scores([r["score"] for r in bilord_results])

    for i, r in enumerate(sap_results): r["norm_score"] = sap_norm[i]
    for i, r in enumerate(bilord_results): r["norm_score"] = bilord_norm[i]

    combined = {}
    for r in sap_results:
        combined[r["code"]] = alpha * r["norm_score"]
    for r in bilord_results:
        combined[r["code"]] = combined.get(r["code"], 0) + beta * r["norm_score"]

    candidate_codes = list(combined.keys())

    # Graph cohesion boost
    graph = build_networkx_graph(root_node)
    cohesion_scores = compute_graph_cohesion(graph, candidate_codes)
    for code in combined:
        combined[code] += gamma * cohesion_scores.get(code, 0)

    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    return ranked

# Example usage
query = "Type 2 diabetes mellitus with neuropathy"
final_results = late_fusion_with_graph_cohesion(query)
print(final_results)


AttributeError: Can't get attribute 'ICDcodeNode' on <module '__main__'>

In [ ]:
#ppr
import pandas as pd
import faiss
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import networkx as nx
import pickle

# ===========================
# CONFIGURATION
# ===========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

# Model names
SAPBERT_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
BILORD_MODEL = "pritamdeka/BioBERT-mnli-snli"  # Example Bilord model

# Paths
SAPBERT_INDEX = r"C:\path\to\sapbert.index"
SAPBERT_META = r"C:\path\to\sapbert_metadata.xlsx"
BILORD_INDEX = r"C:\path\to\bilord.index"
BILORD_META = r"C:\path\to\bilord_metadata.xlsx"
GRAPH_PKL = r"C:\path\to\icd_code_hierarchy.pkl"

# ===========================
# LOAD MODELS & INDEXES
# ===========================
sap_tokenizer = AutoTokenizer.from_pretrained(SAPBERT_MODEL)
sap_model = AutoModel.from_pretrained(SAPBERT_MODEL).to(DEVICE).eval()

bilord_tokenizer = AutoTokenizer.from_pretrained(BILORD_MODEL)
bilord_model = AutoModel.from_pretrained(BILORD_MODEL).to(DEVICE).eval()

sap_index = faiss.read_index(SAPBERT_INDEX)
bilord_index = faiss.read_index(BILORD_INDEX)

sap_meta_df = pd.read_excel(SAPBERT_META)
bilord_meta_df = pd.read_excel(BILORD_META)

sap_metadata = {i: {"code": row["Code"], "description": row["Description"]} for i, row in sap_meta_df.iterrows()}
bilord_metadata = {i: {"code": row["Code"], "description": row["Description"]} for i, row in bilord_meta_df.iterrows()}

# ===========================
# LOAD ICD GRAPH
# ===========================
with open(GRAPH_PKL, "rb") as f:
    node_dict = pickle.load(f)

root_node = node_dict['root']

def build_networkx_graph(root_node):
    graph = nx.Graph()
    def add_nodes_edges(node):
        graph.add_node(node.icd_code, description=node.description)
        for child in node.get_children():
            graph.add_edge(node.icd_code, child.icd_code)
            add_nodes_edges(child)
    add_nodes_edges(root_node)
    return graph

graph = build_networkx_graph(root_node)
print(f"Graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

# ===========================
# EMBEDDING FUNCTIONS
# ===========================
def mean_pooling(last_hidden_state, attention_mask):
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)

def embed_texts(texts, tokenizer, model):
    all_embs = []
    with torch.no_grad():
        for i in range(0, len(texts), BATCH_SIZE):
            batch = texts[i:i+BATCH_SIZE]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
            out = model(**enc)
            emb = mean_pooling(out.last_hidden_state, enc["attention_mask"])
            emb = emb.cpu().numpy()
            all_embs.append(emb)
    embs = np.vstack(all_embs).astype("float32")
    embs /= np.linalg.norm(embs, axis=1, keepdims=True)  # L2 normalization
    return embs

# ===========================
# SEARCH FUNCTIONS
# ===========================
def normalize_scores(scores):
    arr = np.array(scores)
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

def search_model(query, index, tokenizer, model, metadata, top_k=10):
    q_emb = embed_texts([query], tokenizer, model)
    D, I = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        meta = metadata[idx]
        results.append({"code": meta["code"], "description": meta["description"], "score": float(score)})
    return results

# ===========================
# GRAPH FUNCTIONS
# ===========================
def compute_ppr_scores(graph, seed_codes):
    personalization = {node: 0 for node in graph.nodes()}
    for code in seed_codes:
        if code in graph:
            personalization[code] = 1.0
    ppr = nx.pagerank(graph, alpha=0.85, personalization=personalization)
    return ppr

# ===========================
# LATE FUSION + PPR PIPELINE
# ===========================
def late_fusion_with_ppr_auto(query, alpha=0.5, beta=0.5, gamma=0.3, top_k=10, ppr_seeds=5):
    # Step 1: Retrieve from both models
    sap_results = search_model(query, sap_index, sap_tokenizer, sap_model, sap_metadata, top_k)
    bilord_results = search_model(query, bilord_index, bilord_tokenizer, bilord_model, bilord_metadata, top_k)

    # Step 2: Normalize scores
    sap_norm = normalize_scores([r["score"] for r in sap_results])
    bilord_norm = normalize_scores([r["score"] for r in bilord_results])
    for i, r in enumerate(sap_results): r["norm_score"] = sap_norm[i]
    for i, r in enumerate(bilord_results): r["norm_score"] = bilord_norm[i]

    # Step 3: Late fusion
    combined = {}
    for r in sap_results:
        combined[r["code"]] = alpha * r["norm_score"]
    for r in bilord_results:
        combined[r["code"]] = combined.get(r["code"], 0) + beta * r["norm_score"]

    # Step 4: Select top-N seeds for PPR
    ranked_initial = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    seed_codes = [code for code, _ in ranked_initial[:ppr_seeds]]

    # Step 5: Compute PPR scores
    ppr_scores = compute_ppr_scores(graph, seed_codes)

    # Normalize PPR for candidate codes
    candidate_codes = list(combined.keys())
    ppr_values = [ppr_scores.get(code, 0) for code in candidate_codes]
    ppr_norm = normalize_scores(ppr_values)

    # Step 6: Boost with PPR
    for i, code in enumerate(candidate_codes):
        combined[code] += gamma * ppr_norm[i]

    # Final ranking
    ranked_final = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    return ranked_final

# ===========================
# TEST PIPELINE
# ===========================
query = "Type 2 diabetes mellitus with neuropathy"
final_results = late_fusion_with_ppr_auto(query)
print("Top 10 ICD codes after late fusion + PPR:")
for code, score in final_results[:10]:
    print(f"{code}: {score:.4f}")


In [7]:

import pandas as pd
import faiss
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

# Model names
SAPBERT_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
BILORD_MODEL = "FremyCompany/BioLORD-2023"

# Paths
SAPBERT_INDEX = r"C:\Users\UNegi\Downloads\icd10_faiss_sapbert_from_pubmedbert.index"
SAPBERT_META = r"C:\Users\UNegi\Downloads\icd10_metadata_sapbert_from_pubmedbert.xlsx"
BILORD_INDEX = r"C:\Users\UNegi\Documents\Project\makethon\icd10_faiss_biolord.index"
BILORD_META = r"C:\Users\UNegi\Documents\Project\makethon\icd10_metadata_biolord.xlsx"

# Load models
sap_tokenizer = AutoTokenizer.from_pretrained(SAPBERT_MODEL)
sap_model = AutoModel.from_pretrained(SAPBERT_MODEL).to(DEVICE).eval()

bilord_tokenizer = AutoTokenizer.from_pretrained(BILORD_MODEL)
bilord_model = AutoModel.from_pretrained(BILORD_MODEL).to(DEVICE).eval()

# Load FAISS indexes
sap_index = faiss.read_index(SAPBERT_INDEX)
bilord_index = faiss.read_index(BILORD_INDEX)

# Load metadata
sap_meta_df = pd.read_excel(SAPBERT_META)
bilord_meta_df = pd.read_excel(BILORD_META)

sap_metadata = {i: {"code": row["Code"], "description": row["Description"]} for i, row in sap_meta_df.iterrows()}
bilord_metadata = {i: {"code": row["Code"], "description": row["Description"]} for i, row in bilord_meta_df.iterrows()}

# Utility functions
def mean_pooling(last_hidden_state, attention_mask):
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)

def embed_texts(texts, tokenizer, model):
    all_embs = []
    with torch.no_grad():
        for i in range(0, len(texts), BATCH_SIZE):
            batch = texts[i:i+BATCH_SIZE]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
            out = model(**enc)
            emb = mean_pooling(out.last_hidden_state, enc["attention_mask"])
            emb = emb.cpu().numpy()
            all_embs.append(emb)
    embs = np.vstack(all_embs).astype("float32")
    embs /= np.linalg.norm(embs, axis=1, keepdims=True)  # L2 normalization
    return embs

def normalize_scores(scores):
    arr = np.array(scores)
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

def search_model(query, index, tokenizer, model, metadata, top_k=100):
    q_emb = embed_texts([query], tokenizer, model)
    D, I = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        meta = metadata[idx]
        results.append({"code": meta["code"], "description": meta["description"], "score": float(score), "id": idx})
    return results

def late_fusion(query, alpha=0.7, beta=0.3, top_k=100):
    sap_results = search_model(query, sap_index, sap_tokenizer, sap_model, sap_metadata, top_k)
    bilord_results = search_model(query, bilord_index, bilord_tokenizer, bilord_model, bilord_metadata, top_k)

    # Normalize scores
    sap_norm = normalize_scores([r["score"] for r in sap_results])
    bilord_norm = normalize_scores([r["score"] for r in bilord_results])

    # Map normalized scores back
    for i, r in enumerate(sap_results): r["norm_score"] = sap_norm[i]
    for i, r in enumerate(bilord_results): r["norm_score"] = bilord_norm[i]

    # Merge and fuse
    combined = {}
    for r in sap_results:
        combined[r["code"]] = alpha * r["norm_score"]
    for r in bilord_results:
        combined[r["code"]] = combined.get(r["code"], 0) + beta * r["norm_score"]

    # Sort by combined score
    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)

    # Convert to desired output format
    formatted_results = []
    for code, score in ranked:
        # Find description from metadata (prefer SapBERT metadata, fallback to BioLORD)
        description = sap_meta_df.loc[sap_meta_df["Code"] == code, "Description"].values
        if len(description) == 0:
            description = bilord_meta_df.loc[bilord_meta_df["Code"] == code, "Description"].values
        description = description[0] if len(description) > 0 else "Description not found"
        formatted_results.append({
            "ICD_Code": code.strip(),
            "Description": description,
            "Score (cosine)": round(score, 4)
        })

    return formatted_results

# Example usage
query = "copd"
final_results = late_fusion(query)
for res in final_results[:100]:
    print(res)


{'ICD_Code': 'J449', 'Description': 'Chronic obstructive pulmonary disease, unspecified', 'Score (cosine)': np.float64(1.0)}
{'ICD_Code': 'J4489', 'Description': 'Other specified chronic obstructive pulmonary disease', 'Score (cosine)': np.float64(0.5662)}
{'ICD_Code': 'J849', 'Description': 'Interstitial pulmonary disease, unspecified', 'Score (cosine)': np.float64(0.5636)}
{'ICD_Code': 'J64', 'Description': 'Unspecified pneumoconiosis', 'Score (cosine)': np.float64(0.4358)}
{'ICD_Code': 'J441', 'Description': 'Chronic obstructive pulmonary disease with (acute) exacerbation', 'Score (cosine)': np.float64(0.4164)}
{'ICD_Code': 'J42', 'Description': 'Unspecified chronic bronchitis', 'Score (cosine)': np.float64(0.404)}
{'ICD_Code': 'J410', 'Description': 'Simple chronic bronchitis', 'Score (cosine)': np.float64(0.3848)}
{'ICD_Code': 'I279', 'Description': 'Pulmonary heart disease, unspecified', 'Score (cosine)': np.float64(0.3489)}
{'ICD_Code': 'J8410', 'Description': 'Pulmonary fibrosi

In [ ]:

import pandas as pd
import faiss
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

# Model names
SAPBERT_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
BILORD_MODEL = "FremyCompany/BioLORD-2023"

# Paths
SAPBERT_INDEX = r"C:\Users\UNegi\Downloads\icd10_faiss_sapbert_from_pubmedbert.index"
SAPBERT_META = r"C:\Users\UNegi\Downloads\icd10_metadata_sapbert_from_pubmedbert.xlsx"
BILORD_INDEX = r"C:\Users\UNegi\Documents\Project\makethon\icd10_faiss_biolord.index"
BILORD_META = r"C:\Users\UNegi\Documents\Project\makethon\icd10_metadata_biolord.xlsx"

# Load models
sap_tokenizer = AutoTokenizer.from_pretrained(SAPBERT_MODEL)
sap_model = AutoModel.from_pretrained(SAPBERT_MODEL).to(DEVICE).eval()

bilord_tokenizer = AutoTokenizer.from_pretrained(BILORD_MODEL)
bilord_model = AutoModel.from_pretrained(BILORD_MODEL).to(DEVICE).eval()

# Load FAISS indexes
sap_index = faiss.read_index(SAPBERT_INDEX)
bilord_index = faiss.read_index(BILORD_INDEX)

# Load metadata
sap_meta_df = pd.read_excel(SAPBERT_META)
bilord_meta_df = pd.read_excel(BILORD_META)

sap_metadata = {i: {"code": row["Code"], "description": row["Description"]} for i, row in sap_meta_df.iterrows()}
bilord_metadata = {i: {"code": row["Code"], "description": row["Description"]} for i, row in bilord_meta_df.iterrows()}

# Utility functions
def mean_pooling(last_hidden_state, attention_mask):
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)

def embed_texts(texts, tokenizer, model):
    all_embs = []
    with torch.no_grad():
        for i in range(0, len(texts), BATCH_SIZE):
            batch = texts[i:i+BATCH_SIZE]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
            out = model(**enc)
            emb = mean_pooling(out.last_hidden_state, enc["attention_mask"])
            emb = emb.cpu().numpy()
            all_embs.append(emb)
    embs = np.vstack(all_embs).astype("float32")
    embs /= np.linalg.norm(embs, axis=1, keepdims=True)  # L2 normalization
    return embs

def normalize_scores(scores):
    arr = np.array(scores)
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

def softmax_normalize(scores):
    arr = np.array(scores)
    exp_arr = np.exp(arr - np.max(arr))  # stability trick
    return exp_arr / (exp_arr.sum() + 1e-8)

def search_model(query, index, tokenizer, model, metadata, top_k=200):
    q_emb = embed_texts([query], tokenizer, model)
    D, I = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        meta = metadata[idx]
        results.append({"code": meta["code"], "description": meta["description"], "score": float(score), "id": idx})
    return results

def late_fusion(query, alpha=0.7, beta=0.3, top_k=100):
    sap_results = search_model(query, sap_index, sap_tokenizer, sap_model, sap_metadata, top_k)
    bilord_results = search_model(query, bilord_index, bilord_tokenizer, bilord_model, bilord_metadata, top_k)

    # Normalize scores
    sap_norm = normalize_scores([r["score"] for r in sap_results])
    bilord_norm = normalize_scores([r["score"] for r in bilord_results])

    # Map normalized scores back
    for i, r in enumerate(sap_results): r["norm_score"] = sap_norm[i]
    for i, r in enumerate(bilord_results): r["norm_score"] = bilord_norm[i]

    # Merge and fuse
    combined = {}
    for r in sap_results:
        combined[r["code"]] = alpha * r["norm_score"]
    for r in bilord_results:
        combined[r["code"]] = combined.get(r["code"], 0) + beta * r["norm_score"]

    # Sort by combined score
    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)

    # Convert to desired output format
    formatted_results = []
    for code, score in ranked:
        # Find description from metadata (prefer SapBERT metadata, fallback to BioLORD)
        description = sap_meta_df.loc[sap_meta_df["Code"] == code, "Description"].values
        if len(description) == 0:
            description = bilord_meta_df.loc[bilord_meta_df["Code"] == code, "Description"].values
        description = description[0] if len(description) > 0 else "Description not found"
        formatted_results.append({
            "ICD_Code": code.strip(),
            "Description": description,
            "Score (cosine)": round(score, 4)
        })

    return formatted_results

# Example usage
query = "copd"
final_results = late_fusion(query)
for res in final_results[:100]:
    print(res)


{'ICD_Code': 'J449', 'Description': 'Chronic obstructive pulmonary disease, unspecified', 'Score (cosine)': np.float64(1.0)}
{'ICD_Code': 'J4489', 'Description': 'Other specified chronic obstructive pulmonary disease', 'Score (cosine)': np.float64(0.5662)}
{'ICD_Code': 'J849', 'Description': 'Interstitial pulmonary disease, unspecified', 'Score (cosine)': np.float64(0.5636)}
{'ICD_Code': 'J64', 'Description': 'Unspecified pneumoconiosis', 'Score (cosine)': np.float64(0.4358)}
{'ICD_Code': 'J441', 'Description': 'Chronic obstructive pulmonary disease with (acute) exacerbation', 'Score (cosine)': np.float64(0.4164)}
{'ICD_Code': 'J42', 'Description': 'Unspecified chronic bronchitis', 'Score (cosine)': np.float64(0.404)}
{'ICD_Code': 'J410', 'Description': 'Simple chronic bronchitis', 'Score (cosine)': np.float64(0.3848)}
{'ICD_Code': 'I279', 'Description': 'Pulmonary heart disease, unspecified', 'Score (cosine)': np.float64(0.3489)}
{'ICD_Code': 'J8410', 'Description': 'Pulmonary fibrosi